# 03_gold_transform

**Purpose:** Build the Gold star schema for the Power BI semantic model and reconcile it with Silver and Bronze.
Reads only Lakehouse tables; the reject and in-scope flags come from `silver_petrinex_vol_flagged`.
The run ID is the `_dq_run_id` of the 02_silver_transform run that wrote the flags, so Silver DQ results and
Gold reconciliation results share one `run_id`.

**Input tables:** `silver_petrinex_vol_flagged`, `silver_petrinex_vol`, `silver_dim_well`, `bronze_petrinex_vol`,
`dq_run_log`.

**Output tables:**
- `gold_dim_well` (overwrite): Silver well dimension plus placeholder rows for producing wells missing from ST37
- `gold_fact_production` (overwrite, partitioned by `production_month`): grain month x facility x well x product;
  PROD activity, WI rows, not rejected
- `gold_dim_date` (overwrite): one row per day from the first to the last production month
- `gold_dim_operator` (overwrite): one row per `OperatorBAID`
- `reconciliation_results` (append); the notebook stops after writing them if any test fails

**Run order:** step 3 of 3: 01_bronze_transform -> 02_silver_transform -> **03_gold_transform**.
Run 02_silver_transform first: this notebook stops if the flag table is missing, out of date, or written by a
02_silver_transform run that did not finish.

In [ ]:
from pyspark.sql import functions as F

flagged = spark.table("silver_petrinex_vol_flagged")
sp = spark.table("silver_petrinex_vol")
dim = spark.table("silver_dim_well")

# Reuse the run ID of the 02_silver_transform run that wrote the flags
run_ids = [r[0] for r in flagged.select("_dq_run_id").distinct().collect()]
if len(run_ids) != 1:
    raise ValueError(f"silver_petrinex_vol_flagged must hold one _dq_run_id, found {run_ids}. "
                     "Run 02_silver_transform first.")
RUN_ID = run_ids[0]

# Stop if Silver was reloaded after the flags were written
n_silver = sp.count()
n_flagged = flagged.count()
if n_flagged != n_silver:
    raise ValueError(f"silver_petrinex_vol_flagged has {n_flagged} rows, silver_petrinex_vol has {n_silver}. "
                     "Run 02_silver_transform first.")

# Stop if that 02_silver_transform run did not finish: dq_run_log is written only after the DQ rule tests pass
if spark.table("dq_run_log").filter(F.col("run_id") == RUN_ID).count() == 0:
    raise ValueError(f"Run {RUN_ID} of 02_silver_transform did not finish (not in dq_run_log). "
                     "Rerun 02_silver_transform and check dq_rule_test_results.")

fact_rows = flagged.filter(~F.col("dq_rejected") & F.col("dq_in_scope"))
print("RUN_ID:", RUN_ID, " silver rows:", n_silver)

In [ ]:
# Gold well dimension: silver dimension plus placeholder rows for wells missing from ST37
missing = (fact_rows.select("well_key").distinct()
    .join(dim.select("well_key"), "well_key", "left_anti"))

placeholders = missing
for c, t in dim.dtypes:
    if c != "well_key":
        placeholders = placeholders.withColumn(c, F.lit(None).cast(t))
placeholders = (placeholders
    .withColumn("in_production_string", F.lit(False))
    .withColumn("in_bh", F.lit(False))
    .withColumn("is_placeholder", F.lit(True))
    .select(dim.columns))

gold_dim = dim.unionByName(placeholders)
gold_dim.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_dim_well")
print("placeholder wells:", placeholders.count())

In [ ]:
# Gold fact table
fact = (fact_rows
    .select("production_month", "well_key", "ProductID", "ReportingFacilityID",
            "OperatorBAID", "OperatorName", "volume", "energy", "hours",
            "submission_date", "_source_file")
    .withColumn("_gold_run_id", F.lit(RUN_ID)))
# Partition by month, one file per month (DEC-023)
(fact.repartition("production_month")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .partitionBy("production_month")
    .saveAsTable("gold_fact_production"))

In [ ]:
# Gold date and operator dimensions, derived from the Gold fact table
fact = spark.table("gold_fact_production")
b = fact.agg(F.min("production_month").alias("lo"), F.max("production_month").alias("hi")).collect()[0]

dim_date = (
    spark.range(1)
    .select(F.explode(F.sequence(F.lit(b["lo"]), F.last_day(F.lit(b["hi"])), F.expr("interval 1 day"))).alias("date"))
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .withColumn("month_start", F.trunc("date", "month"))
)
dim_date.write.mode("overwrite").format("delta").saveAsTable("gold_dim_date")

# Optional: operator dimension (one row per BAID)
dim_operator = (
    fact.filter(F.col("OperatorBAID").isNotNull())
    .groupBy("OperatorBAID")
    .agg(F.max("OperatorName").alias("operator_name"))
)
dim_operator.write.mode("overwrite").format("delta").saveAsTable("gold_dim_operator")

In [ ]:
# Reconciliation tests
g = spark.table("gold_fact_production")
gd = spark.table("gold_dim_well")
n_rejected = flagged.filter(F.col("dq_rejected")).count()
n_out_of_scope = flagged.filter(~F.col("dq_rejected") & ~F.col("dq_in_scope")).count()

grain = ["production_month", "ReportingFacilityID", "well_key", "ProductID"]
dup_grain = g.groupBy(*grain).count().filter("count > 1").count()
orphans = g.join(gd.select("well_key"), "well_key", "left_anti").count()

silver_sum = fact_rows.groupBy("ProductID").agg(F.sum("volume").alias("s"))
gold_sum = g.groupBy("ProductID").agg(F.sum("volume").alias("g"))
sum_mismatch = (silver_sum.join(gold_sum, "ProductID", "full_outer")
    .filter(F.abs(F.coalesce("s", F.lit(0)) - F.coalesce("g", F.lit(0))) > 0.001)
    .count())

tests = [
    ("bronze_equals_silver_rows", spark.table("bronze_petrinex_vol").count(), n_silver),
    ("silver_equals_rejected_plus_out_of_scope_plus_gold", n_silver, n_rejected + n_out_of_scope + g.count()),
    ("gold_volume_totals_match_silver_by_product", 0, sum_mismatch),
    ("gold_fact_grain_unique", 0, dup_grain),
    ("gold_fact_no_orphan_wells", 0, orphans),
]
rows = [(RUN_ID, n, int(e), int(a), "PASS" if e == a else "FAIL") for n, e, a in tests]
schema = "run_id string, test_name string, expected long, actual long, status string"
spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable("reconciliation_results")

# Stop on a failed test, after the results are stored
failed = [r for r in rows if r[4] == "FAIL"]
if failed:
    display(spark.createDataFrame(failed, schema))
    detail = "; ".join(f"{name} (expected {exp}, actual {act})" for _, name, exp, act, _ in failed)
    raise ValueError(f"Reconciliation failed for run {RUN_ID}: {len(failed)} of {len(rows)} tests FAIL: {detail}. "
                     "See reconciliation_results. The Gold tables were already overwritten by this run.")

In [ ]:
# Summary of this run
display(spark.table("reconciliation_results").filter(F.col("run_id") == RUN_ID))
for t in ["gold_fact_production", "gold_dim_well", "gold_dim_date", "gold_dim_operator"]:
    print(t, "rows:", spark.table(t).count())